# Spatial Clue Inference Visualization

This notebook loads a trained `SpatialClueAlignment` checkpoint and runs image-only inference to show:
1. Diagnosis classification
2. Chaos classification
3. Clue segmentation with Grad-CAM overlays and ground-truth mask comparison

In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve().parent
KGCL_ROOT = PROJECT_ROOT / "KGCL"
for path in (PROJECT_ROOT, KGCL_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from KGCL.datasets.constants import CHAOS_LABELS, CLUE_NAMES, IDX_TO_DIAGNOSIS
from KGCL.datasets.transforms import DataTransforms
from KGCL.models.kgcl.segmenatation_module import SpatialClueAlignment

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
CHECKPOINT_PATH = PROJECT_ROOT / "KGCL" / "checkpoints" / "joint_spatial_clue" / "YOUR_RUN" / "last.ckpt"
IMAGE_PATH = PROJECT_ROOT / "Annotated_data" / "Images" / "ISIC_0000000.jpg"
GROUND_TRUTH_MASK_DIR = PROJECT_ROOT / "Annotated_data" / "GroundTruthMasks"
IMG_ENCODER = "vit_base"  # "resnet_50" or "vit_base"
MASK_THRESHOLD = 0.5

assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"
assert IMAGE_PATH.exists(), f"Image not found: {IMAGE_PATH}"

In [ ]:
def build_model(checkpoint_path: Path, img_encoder: str):
    model = SpatialClueAlignment.load_from_checkpoint(
        str(checkpoint_path),
        map_location=device,
        img_encoder=img_encoder,
        freeze_bert=True,
    )
    model.eval()
    model.to(device)
    return model


def get_target_module(model, img_encoder: str):
    if "resnet" in img_encoder:
        return model.img_encoder_q.model.layer4[-1]
    if "vit" in img_encoder:
        return model.img_encoder_q.model.blocks[-1].norm1
    raise ValueError(f"Unsupported encoder: {img_encoder}")


def run_image_only_forward(model, imgs: torch.Tensor):
    img_feat_q, patch_feat_q = model.img_encoder_q(imgs)
    img_feat_raw = model.get_global_image_feature_for_cls(img_feat_q)
    patch_feat_map = model.to_spatial_map(patch_feat_q)

    diagnosis_logits = model.diagnosis_head(img_feat_raw)
    chaos_logits = model.chaos_head(img_feat_raw)
    clue_seg_logits = model.clue_seg_head(patch_feat_map, output_size=imgs.shape[-2:])

    return {
        "diagnosis_logits": diagnosis_logits,
        "chaos_logits": chaos_logits,
        "clue_seg_logits": clue_seg_logits,
    }


def denormalize_image(tensor: torch.Tensor) -> np.ndarray:
    image = tensor.detach().cpu().squeeze(0).permute(1, 2, 0).numpy()
    image = (image * 0.5) + 0.5
    return np.clip(image, 0.0, 1.0)


def overlay_heatmap(image: np.ndarray, heatmap: np.ndarray, alpha: float = 0.4) -> np.ndarray:
    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_rgb = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap_rgb, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return np.clip((1 - alpha) * image + alpha * heatmap_rgb, 0.0, 1.0)


def normalize_map(array: np.ndarray) -> np.ndarray:
    array = array.astype(np.float32)
    array = array - array.min()
    return array / (array.max() + 1e-8)


def load_ground_truth_masks(mask_dir: Path, image_path: Path):
    mask_path = mask_dir / f"{image_path.stem}_mask.npy"
    if not mask_path.exists():
        mask_path = mask_dir / f"{image_path.stem}_mask11.npy"
    assert mask_path.exists(), f"Ground-truth mask not found for {image_path.stem}"
    masks = np.load(mask_path).astype(np.float32)
    if masks.ndim == 2:
        masks = masks[None, ...]
    return mask_path, masks


model = build_model(CHECKPOINT_PATH, IMG_ENCODER)
target_module = get_target_module(model, IMG_ENCODER)
transform = DataTransforms(is_train=False, crop_size=224)
pil_image = Image.open(IMAGE_PATH).convert("RGB")
input_tensor = transform(pil_image).unsqueeze(0).to(device)
base_image = denormalize_image(input_tensor)
gt_mask_path, gt_masks = load_ground_truth_masks(GROUND_TRUTH_MASK_DIR, IMAGE_PATH)

display(pil_image)
print("Ground-truth masks:", gt_mask_path.name)
print("Input tensor shape:", tuple(input_tensor.shape))

In [ ]:
class GradCAM:
    def __init__(self, model, target_module, img_encoder: str):
        self.model = model
        self.target_module = target_module
        self.img_encoder = img_encoder
        self.activations = None
        self.gradients = None
        self.forward_handle = target_module.register_forward_hook(self._save_activation)
        self.backward_handle = target_module.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inputs, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

    def _make_cam(self, x: torch.Tensor) -> np.ndarray:
        if "resnet" in self.img_encoder:
            weights = self.gradients.mean(dim=(2, 3), keepdim=True)
            cam = (weights * self.activations).sum(dim=1, keepdim=True)
        else:
            activations = self.activations[:, 1:, :]
            gradients = self.gradients[:, 1:, :]
            side = int(activations.shape[1] ** 0.5)
            weights = gradients.mean(dim=1, keepdim=True)
            cam = (weights * activations).sum(dim=2).reshape(-1, 1, side, side)

        cam = F.relu(cam)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        return normalize_map(cam)

    def diagnosis_cam(self, x: torch.Tensor, target_class: int | None = None):
        self.model.zero_grad(set_to_none=True)
        outputs = run_image_only_forward(self.model, x)
        logits = outputs["diagnosis_logits"]
        pred_idx = int(logits.argmax(dim=1).item())
        class_idx = pred_idx if target_class is None else int(target_class)
        logits[:, class_idx].sum().backward(retain_graph=True)
        return self._make_cam(x), pred_idx, logits.softmax(dim=1).detach().cpu().numpy()[0]

    def chaos_cam(self, x: torch.Tensor, chaos_idx: int):
        self.model.zero_grad(set_to_none=True)
        outputs = run_image_only_forward(self.model, x)
        logits = outputs["chaos_logits"]
        logits[:, chaos_idx].sum().backward(retain_graph=True)
        probs = torch.sigmoid(logits).detach().cpu().numpy()[0]
        return self._make_cam(x), probs

    def clue_cam(self, x: torch.Tensor, clue_idx: int):
        self.model.zero_grad(set_to_none=True)
        outputs = run_image_only_forward(self.model, x)
        clue_logits = outputs["clue_seg_logits"]
        clue_logits[:, clue_idx].mean().backward(retain_graph=True)
        clue_probs = torch.sigmoid(clue_logits).detach().cpu().numpy()[0]
        return self._make_cam(x), clue_probs[clue_idx]


grad_cam = GradCAM(model, target_module, IMG_ENCODER)

In [ ]:
with torch.no_grad():
    outputs = run_image_only_forward(model, input_tensor)

diagnosis_logits = outputs["diagnosis_logits"]
chaos_logits = outputs["chaos_logits"]
clue_seg_logits = outputs["clue_seg_logits"]

diagnosis_probs = F.softmax(diagnosis_logits, dim=1).squeeze(0).cpu().numpy()
diagnosis_idx = int(diagnosis_probs.argmax())
chaos_probs = torch.sigmoid(chaos_logits).squeeze(0).cpu().numpy()
clue_probs = torch.sigmoid(clue_seg_logits).squeeze(0).cpu().numpy()
clue_preds = (clue_probs >= MASK_THRESHOLD).astype(np.float32)

print(f"Diagnosis prediction: {IDX_TO_DIAGNOSIS[diagnosis_idx]}")
for idx, prob in enumerate(diagnosis_probs):
    print(f"  {IDX_TO_DIAGNOSIS[idx]}: {prob:.4f}")

print("Chaos classification:")
for label, prob in zip(CHAOS_LABELS, chaos_probs):
    print(f"  {label}: {prob:.4f}")

In [ ]:
diagnosis_cam, _, _ = grad_cam.diagnosis_cam(input_tensor)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(base_image)
axes[0].set_title(f"Diagnosis: {IDX_TO_DIAGNOSIS[diagnosis_idx]}")
axes[0].axis("off")
axes[1].imshow(overlay_heatmap(base_image, diagnosis_cam))
axes[1].set_title("Diagnosis Grad-CAM")
axes[1].axis("off")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(CHAOS_LABELS), figsize=(6 * len(CHAOS_LABELS), 5))
axes = np.atleast_1d(axes)
for idx, label in enumerate(CHAOS_LABELS):
    chaos_cam, _ = grad_cam.chaos_cam(input_tensor, idx)
    axes[idx].imshow(overlay_heatmap(base_image, chaos_cam))
    axes[idx].set_title(f"{label}\nprob={chaos_probs[idx]:.3f}")
    axes[idx].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
clue_keys = list(CLUE_NAMES.keys())
n_cols = 4
n_rows = int(np.ceil(len(clue_keys) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows))
axes = np.atleast_1d(axes).ravel()

for idx, clue_key in enumerate(clue_keys):
    clue_name = CLUE_NAMES[clue_key]
    clue_cam, clue_prob_map = grad_cam.clue_cam(input_tensor, idx)
    pred_mask = clue_preds[idx]
    gt_mask = gt_masks[idx] if idx < gt_masks.shape[0] else np.zeros_like(pred_mask)
    gt_mask = cv2.resize(gt_mask.astype(np.float32), (pred_mask.shape[1], pred_mask.shape[0]), interpolation=cv2.INTER_NEAREST)

    canvas = np.concatenate(
        [
            overlay_heatmap(base_image, normalize_map(clue_prob_map)),
            overlay_heatmap(base_image, clue_cam),
            overlay_heatmap(base_image, pred_mask),
            overlay_heatmap(base_image, gt_mask),
        ],
        axis=1,
    )
    axes[idx].imshow(canvas)
    axes[idx].set_title(
        f"{clue_name}\nseg={clue_prob_map.max():.3f} | pred_px={int(pred_mask.sum())} | gt_px={int(gt_mask.sum())}"
    )
    axes[idx].axis("off")

for ax in axes[len(clue_keys):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Per-clue panels are ordered left-to-right as: segmentation heatmap, Grad-CAM, predicted mask, ground-truth mask.")

In [ ]:
# Run this when you are finished with the notebook session.
grad_cam.remove()